# Video Clone — Colab full worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daokimluc/video-clone-colab/blob/main/video_clone_colab_backend.ipynb)

**Runtime → GPU → Run all.** Copy `REMOTE_URL` + `SHARED_SECRET` vào app Remote.

ASR / OCR / TTS / Translate trên Colab · Export MP4 trên desktop.

## 1) Install

In [ ]:
!pip -q install fastapi uvicorn httpx faster-whisper edge-tts deep-translator easyocr

import shutil, os

def ensure_cloudflared():
    if shutil.which('cloudflared'):
        print('cloudflared OK'); return
    !wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /tmp/cloudflared
    !cp /tmp/cloudflared /usr/local/bin/cloudflared

ensure_cloudflared()
# force-refresh worker from GitHub (no cache)
!wget -q -O /content/colab_worker.py "https://raw.githubusercontent.com/daokimluc/video-clone-colab/main/colab_worker.py?$(date +%s)"
print('worker size', os.path.getsize('/content/colab_worker.py'))
try:
    import torch
    print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except Exception as e:
    print('torch:', e)

## 2) Secret

In [ ]:
import secrets, os
SHARED_SECRET = secrets.token_urlsafe(16)
os.environ['VC_COLAB_SECRET'] = SHARED_SECRET
os.environ['SHARED_SECRET'] = SHARED_SECRET
print('SHARED_SECRET =', SHARED_SECRET)

## 3) Start worker (health only — capabilities optional, never block)

In [ ]:
import os, time, socket, subprocess, httpx, importlib.util, sys

WORKER_PORT = 8765
os.environ['VC_WORKER_PORT'] = str(WORKER_PORT)
os.environ['VC_COLAB_SECRET'] = SHARED_SECRET

def free_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        busy = s.connect_ex(('127.0.0.1', port)) == 0
    if not busy:
        print('Port free', port); return
    print('Killing port', port)
    subprocess.run(['bash','-lc', f'fuser -k {port}/tcp || true'], check=False)
    time.sleep(1.2)

free_port(WORKER_PORT)

# load worker file
spec = importlib.util.spec_from_file_location('colab_worker', '/content/colab_worker.py')
mod = importlib.util.module_from_spec(spec)
sys.modules['colab_worker'] = mod
spec.loader.exec_module(mod)
mod.SHARED_SECRET = SHARED_SECRET
mod.WORKER_PORT = WORKER_PORT

import uvicorn, threading

def run():
    # single worker process, but do not import heavy libs at startup
    uvicorn.run(mod.app, host='127.0.0.1', port=WORKER_PORT, log_level='warning')

t = threading.Thread(target=run, daemon=True)
t.start()

ok = False
for i in range(20):
    try:
        # short timeout — health must be instant
        r = httpx.get(
            f'http://127.0.0.1:{WORKER_PORT}/health',
            headers={'X-VC-Secret': SHARED_SECRET},
            timeout=2.0,
        )
        print('Local health:', r.status_code, r.text[:200])
        if r.status_code == 200:
            ok = True
            break
    except Exception as e:
        print(f'  wait {i+1}/20…', type(e).__name__)
        time.sleep(0.5)

if not ok:
    raise RuntimeError('Worker health failed — Runtime → Restart session, Run all')

# capabilities optional (must not hang the notebook)
try:
    c = httpx.get(
        f'http://127.0.0.1:{WORKER_PORT}/capabilities',
        headers={'X-VC-Secret': SHARED_SECRET},
        timeout=5.0,
    )
    print('Capabilities:', c.status_code, c.text[:300])
except Exception as e:
    print('Capabilities skip (OK to continue):', e)

print('Worker ready on port', WORKER_PORT, '→ next: Tunnel cell')

## 4) Tunnel

In [ ]:
import re, subprocess, time, httpx

if globals().get('_VC_TUNNEL_PROC'):
    try: _VC_TUNNEL_PROC.terminate()
    except: pass
subprocess.run(['bash','-lc','pkill -f "cloudflared tunnel" || true'], check=False)
time.sleep(0.5)

cmd=['cloudflared','tunnel','--url',f'http://127.0.0.1:{WORKER_PORT}','--no-autoupdate']
print('Starting:', ' '.join(cmd), flush=True)
_VC_TUNNEL_PROC=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

REMOTE_URL=None
pat=re.compile(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com')
deadline=time.time()+90
while time.time()<deadline and not REMOTE_URL:
    line=_VC_TUNNEL_PROC.stdout.readline()
    if not line:
        if _VC_TUNNEL_PROC.poll() is not None: break
        continue
    line=line.rstrip(); print(line, flush=True)
    m=pat.search(line)
    if m: REMOTE_URL=m.group(0).rstrip('/')

if not REMOTE_URL:
    raise RuntimeError('No tunnel URL')

print('='*60)
print('REMOTE_URL =', REMOTE_URL)
print('SHARED_SECRET =', SHARED_SECRET)
print('='*60)
time.sleep(2)
try:
    r=httpx.get(REMOTE_URL+'/health', headers={'X-VC-Secret':SHARED_SECRET}, timeout=20, follow_redirects=True)
    print('Public health:', r.status_code, r.text[:200])
except Exception as e:
    print('Public health later (DNS):', e)
print('Keep runtime running. Desktop: mode Remote + paste URL & secret.')